In [1]:
import time
from pynput.keyboard import Key, Controller, Listener
import random
import mss
import cv2
import win32gui
import win32com
import win32com.client
import ctypes
import pygetwindow as gw
import pyautogui
from PIL import Image
from screeninfo import get_monitors

from detection import *
from exp_processor import ExpProcessor

# --- PyNput 相關設定 (保持不變) ---
keyboard = Controller()
# pause = False
currently_pressed = set()  # 記錄目前被按下的鍵

def safe_press(key):
    # if key not in currently_pressed:
    keyboard.press(key)
    currently_pressed.add(key)

def safe_release(key):
    if key in currently_pressed:
        keyboard.release(key)
        currently_pressed.remove(key)

def safe_release_all():
    for key in list(currently_pressed): # 迭代副本以避免在迭代時修改集合
        keyboard.release(key)
    currently_pressed.clear()

def wait_with_pause(duration):
    global pause
    start_wait_time = time.time()
    while (time.time() - start_wait_time) < duration:
        if pause:
            print("[暫停中] 釋放所有按鍵")
            safe_release_all()
            while pause:
                time.sleep(0.1) # 暫停時檢查頻率
            print("[恢復]")
            # 由於暫停會中斷計時，這裡需要調整已等待的時間，或者直接中斷當前等待，重新開始移動邏輯
            # 為了簡單起見，我們讓它直接中斷，回到主迴圈重新判斷
            return False # 表示等待未完成，因為被暫停了
        time.sleep(0.01) # 檢查頻率
    return True # 表示等待完成

def on_press(key):
    global pause
    try:
        if key == Key.f8:
            pause = not pause
            print(f"[狀態切換] {'暫停中' if pause else '繼續運行'}")
            time.sleep(0.5)
    except AttributeError:
        # 非特殊按鍵，例如 'a' 'b' 等
        pass

def click_press(key, duration = 0):
    safe_press(key)
    if duration > 0:
        wait_with_pause(duration)
    safe_release(key)

def attack_during_walk(walk_key, attack_key, duration):
    start_walk = False
    current = time.time()
    while time.time() - current < duration:
        if not start_walk:
            safe_press(walk_key)
            # keyboard.press(walk_key)
            start_walk = True
        # click_press(attack_key)
        safe_press(attack_key)
        time.sleep(0.05)
        safe_release(attack_key)
        time.sleep(0.05)
    safe_release(walk_key)

In [37]:
def get_maple_window(maple_name = "MapleStory Worlds-Artale (?????)"):
    return win32gui.FindWindow(None, maple_name)

def is_maple_found():
    if get_maple_window() == 0:
        return False
    else:
        return True

def is_maple_active(maple_name = "MapleStory Worlds-Artale (?????)"):
    window_id = win32gui.FindWindow(None, maple_name)
    foreground_id = win32gui.GetForegroundWindow()
    if foreground_id == window_id:
        return True
    else:
        return False
    
def show_maple():
    if is_maple_found() is False:
        return
    else:
        shell = win32com.client.Dispatch("WScript.Shell")  # not sure if helps
        shell.SendKeys('%')
        # win32gui.SetForegroundWindow(try_get_window())
        ctypes.windll.user32.SetForegroundWindow(get_maple_window()) 
        ctypes.windll.user32.ShowWindow(get_maple_window(), 9)  # SW_RESTORE = 9
        safe_release_all()

# --- 截圖與偵測函數 (從之前的程式碼複製過來，並將 debug 邏輯移到 main) ---
def capture_window_screenshot(window_title):
    window = gw.getWindowsWithTitle(window_title)
    if len(window) == 0:
        return None, None
    window = window[0]
    left, top, right, bottom = window.left, window.top, window.right, window.bottom
    width = right - left
    height = bottom - top
    monitors = get_monitors()
    for monitor_info in monitors: # 改名避免與 mss.monitor 衝突
        if left >= monitor_info.x and right <= monitor_info.x + monitor_info.width and bottom >= monitor_info.y and top <= monitor_info.y + monitor_info.height:
            try: 
                with mss.mss() as sct:
                    monitor = {"left": left, "top": top, "width": width, "height": height}
                    screenshot = sct.grab(monitor)
                    np_img = np.array(screenshot)
                    np_img = cv2.cvtColor(np_img, cv2.COLOR_BGRA2BGR)
                    img = Image.frombytes("RGB", screenshot.size, screenshot.rgb)
                    return img, np_img
            except Exception as e:
                print(e)
    return None, None

def get_character():
    window_title = 'MapleStory Worlds-Artale (?????)'

    # 小地圖的固定座標 (需要手動測量) - 這些值你需要根據你的實際情況調整
    MINIMAP_X_OFFSET = 20 # 根據你的螢幕截圖，小地圖左上角的x座標
    MINIMAP_Y_OFFSET = 171 # 小地圖左上角的y座標
    MINIMAP_WIDTH = 229   # 小地圖的寬度
    MINIMAP_HEIGHT = 145  # 小地圖的高度

    minimap_character_template_folder = 'assets/minimap_character/' 
    character_centers_minimap = None
    pil_img, np_img = capture_window_screenshot(window_title)
    if np_img is None:
        print(f"找不到視窗 '{window_title}'，請確保遊戲正在運行並位於螢幕上。")
        time.sleep(2)
        # continue
    minimap_end_x = MINIMAP_X_OFFSET + MINIMAP_WIDTH
    minimap_end_y = MINIMAP_Y_OFFSET + MINIMAP_HEIGHT
    minimap_end_x = min(minimap_end_x, np_img.shape[1])
    minimap_end_y = min(minimap_end_y, np_img.shape[0])
    minimap_img = np_img[MINIMAP_Y_OFFSET:minimap_end_y, MINIMAP_X_OFFSET:minimap_end_x]
    character_centers_minimap = detect_character_on_minimap(minimap_img, 
                                                    templates_folder=minimap_character_template_folder, 
                                                    threshold=0.75,)
    display_minimap_img = minimap_img.copy() 
    if character_centers_minimap:
        char_minimap_center_x, char_minimap_center_y = character_centers_minimap[0]
        return char_minimap_center_x, char_minimap_center_y
    return -1, -1

def get_exp(exp_processor, debug = False):
    window_title = 'MapleStory Worlds-Artale (?????)'
    pil_img, np_img = capture_window_screenshot(window_title)
    recent_exp_gain = None
    original_width = pil_img.width
    original_height = pil_img.height

    # 應用偏移量
    width_offset = 22
    height_offset = 56
    # 如果寬度是 10 的倍數，偏移量為 0
    if original_width % 10 == 0:
        width_offset = 0
        height_offset = 0

    # 計算應用偏移量後的有效尺寸，用於長寬比匹配
    current_width = original_width - width_offset
    current_height = original_height - height_offset
    # 獲取匹配到的參考解析度和原始裁切區域
    reference_width, reference_height = 1920,1080
    original_exp_region = (1028, 996, 1190, 1025)

    # 根據應用偏移量後的尺寸和參考比例，計算新的裁切座標 (相對於應用偏移量後的左上角)
    try:
        # 計算原始區域相對於參考解析度的比例
        exp_left_prop = original_exp_region[0] / reference_width
        exp_top_prop = original_exp_region[1] / reference_height
        exp_right_prop = original_exp_region[2] / reference_width
        exp_bottom_prop = original_exp_region[3] / reference_height


        # 根據應用偏移量後的尺寸和計算出的比例，計算新的裁切座標
        # 然後再加上偏移量，得到相對於原始螢幕左上角的座標
        new_exp_left = int(current_width * exp_left_prop) + width_offset
        new_exp_top = int(current_height * exp_top_prop) + height_offset
        new_exp_right = int(current_width * exp_right_prop) + width_offset
        new_exp_bottom = int(current_height * exp_bottom_prop) + height_offset

        # 確保裁切座標在原始圖片範圍內且有效 (right > left, bottom > top)
        new_exp_left = max(0, new_exp_left)
        new_exp_top = max(0, new_exp_top)
        new_exp_right = min(original_width, new_exp_right) # 使用原始寬度進行邊界檢查
        new_exp_bottom = min(original_height, new_exp_bottom) # 使用原始高度進行邊界檢查
        width_diff = current_width - reference_width
        height_diff = current_height - reference_height

        # 計算調整量，與尺寸差異成反比，並限制在 +/- 30 範圍內
        # 這裡使用一個簡單的線性映射，例如每差 100 像素調整 1 像素
        adjustment_factor = 0.01 # 每 100 像素差異調整 1 像素 (1/100 = 0.01)
        max_width_adjustment = 30 # 最大調整量為 +/- 30 像素

        # 寬度調整
        raw_adj_x = width_diff * adjustment_factor
        adj_x = int(max(-max_width_adjustment, min(max_width_adjustment, raw_adj_x)))

        # 應用調整到裁切座標
        adjusted_exp_left = new_exp_left + adj_x

        # 確保調整後的裁切座標在原始圖片範圍內且有效 (right > left, bottom > top)
        new_exp_left = max(0, adjusted_exp_left)

        exp_cropped_image = pil_img.crop((adjusted_exp_left, new_exp_top, new_exp_right, new_exp_bottom))
        recent_exp_gain, exp_processed_image = exp_processor.process_exp_value(exp_cropped_image, n=2)
        # cv2.imshow("Exp image (Press 'q' to quit, any other key to continue)", exp_processed_image)
        # key = cv2.waitKey(1) # 等待1毫秒，讓視窗有時間更新
        # if key == ord('q'): 
        #     cv2.destroyAllWindows()
        #     print("使用者按下 'q' 鍵，程式結束。")
        #=============================================
        # return recent_exp_gain, exp_processed_image
    except Exception as e:
        # 裁切失敗時打印錯誤信息
        print(f"根據比例計算或裁切圖片時發生錯誤: {e}")
    if debug:
        return recent_exp_gain, exp_processed_image
    return recent_exp_gain

def get_enemy():
    window_title = 'MapleStory Worlds-Artale (?????)'

    # 小地圖的固定座標 (需要手動測量) - 這些值你需要根據你的實際情況調整
    MINIMAP_X_OFFSET = 20 # 根據你的螢幕截圖，小地圖左上角的x座標
    MINIMAP_Y_OFFSET = 171 # 小地圖左上角的y座標
    MINIMAP_WIDTH = 229   # 小地圖的寬度
    MINIMAP_HEIGHT = 145  # 小地圖的高度

    minimap_character_template_folder = 'assets/minimap_other_character/' 
    character_centers_minimap = None
    pil_img, np_img = capture_window_screenshot(window_title)
    if np_img is None:
        print(f"找不到視窗 '{window_title}'，請確保遊戲正在運行並位於螢幕上。")
        time.sleep(2)
        # continue
    minimap_end_x = MINIMAP_X_OFFSET + MINIMAP_WIDTH
    minimap_end_y = MINIMAP_Y_OFFSET + MINIMAP_HEIGHT
    minimap_end_x = min(minimap_end_x, np_img.shape[1])
    minimap_end_y = min(minimap_end_y, np_img.shape[0])
    minimap_img = np_img[MINIMAP_Y_OFFSET:minimap_end_y, MINIMAP_X_OFFSET:minimap_end_x]

    character_centers_minimap = detect_red_dots(minimap_img, 
                    templates_folder=minimap_character_template_folder, 
                    threshold=0.75)
    return character_centers_minimap

def goto_freemarket():
    global pause
    pause = True
    safe_release_all()
    print(f"怕怕...暫停一下")
    time.sleep(22)
    button = 'left'
    clicks = 2
    x_coord = 1550 # 2581
    y_coord = 2040 # 1766

    pyautogui.moveTo(x_coord, y_coord, duration=0.5)
    print(f"[{time.time()}] 自由市場GOGO")
    # 點擊滑鼠
    pyautogui.click(x=x_coord, y=y_coord, clicks=clicks, interval=0.25, button=button)

def blue_dragon_loop():
    exp_processor = ExpProcessor()
    global pause
    pause = True
    start_time = time.time()
    current_time = start_time
    skill_time = current_time - 300
    CHARACTER_Y = 91
    CHARACTER_X_L = 66
    CHARACTER_X_R = 161
    while True:
        if pause:
            time.sleep(0.1)
            continue
        click_press('c', duration=3)
        end_time = time.time()
        current_state_time = (end_time - current_time)
        if current_state_time > 12 and not pause:
            walk_time = float(random.randint(20, 30))/10
            px, py = get_character()
            exp_gain = get_exp(exp_processor)
            if exp_gain != None:
                recent_exp_gain, exp_num_str, exp_pct_str = exp_gain[0], exp_gain[1], exp_gain[2]
                print(f"最近經驗:{recent_exp_gain}({exp_num_str}, {exp_pct_str}%)")
                if recent_exp_gain < 4000 and end_time - start_time > 180:
                    goto_freemarket()
                    continue
            if len(get_enemy()) > 0:
                print("有紅點!!!")
                goto_freemarket()
                continue
            # not move right
            if abs(py - CHARACTER_Y) > 5 or (px - CHARACTER_X_L) <= 20:
                attack_during_walk(Key.right, 'c', walk_time)
            attack_during_walk(Key.left, 'c', walk_time + 0.25)
            click_press(Key.right, 0.1)
            time.sleep(0.5)
            click_press('h', 0.1)
            current_time = time.time()
            if current_time - skill_time > 300:
                click_press('a', 0.5)
                time.sleep(0.1)
                skill_time = current_time


def egg_dragon_loop():
    exp_processor = ExpProcessor()
    global pause
    pause = True
    start_time = time.time()
    current_time = start_time
    skill_time = current_time - 300
    CHARACTER_Y = 91
    CHARACTER_X_L = 66
    CHARACTER_X_R = 161
    while True:
        if pause:
            time.sleep(0.1)
            continue
        click_press('c', duration=3)
        end_time = time.time()
        current_state_time = (end_time - current_time)
        if current_state_time > 15 and not pause:
            walk_time = float(random.randint(40, 50))/10
            # px, py = get_character()
            exp_gain = get_exp(exp_processor)
            if exp_gain != None:
                recent_exp_gain, exp_num_str, exp_pct_str = exp_gain[0], exp_gain[1], exp_gain[2]
                print(f"最近經驗:{recent_exp_gain}({exp_num_str}, {exp_pct_str}%)")
                if recent_exp_gain < 4000 and end_time - start_time > 180:
                    goto_freemarket()
                    continue
            if len(get_enemy()) > 0:
                print("有紅點!!!")
                goto_freemarket()
                continue
            # not move right
            # if abs(py - CHARACTER_Y) > 5 or (px - CHARACTER_X_L) <= 20:
            #     attack_during_walk(Key.left, 'c', walk_time)
            attack_during_walk(Key.left, 'c', walk_time)
            attack_during_walk(Key.right, 'c', walk_time + 0.25)
            click_press(Key.left, 0.1)
            time.sleep(0.5)
            click_press('h', 0.1)
            current_time = time.time()
            if current_time - skill_time > 300:
                click_press('a', 0.5)
                time.sleep(0.1)
                skill_time = current_time


In [ ]:
# ============================================================================
# HUMAN-LIKE LAYER  (thin timing/rhythm layer over the existing patrol)
# ----------------------------------------------------------------------------
# This cell REDEFINES blue_dragon_loop() / egg_dragon_loop() so they shadow the
# originals above. All game logic (EXP check, red-dot escape, F8 pause, patrol
# bounds, goto_freemarket) is preserved. Everything is gated behind HUMAN so you
# can tune or disable any piece. Set HUMAN["enabled"] = False to fall straight
# back to the original deterministic behavior.
#
# See auto_train/HUMANIZE_HANDOFF.md for the design rationale.
# ============================================================================
import random
import time
from pynput.keyboard import Key

HUMAN = {
    "enabled": True,

    # --- jittered timing primitive (§5.1) -----------------------------------
    "sleep_spread": 0.25,          # gaussian std as a fraction of the base delay

    # --- irregular attack rhythm (§5.2) -------------------------------------
    "attack_press_base": 0.06,     # was a fixed 0.05 press
    "attack_press_spread": 0.35,
    "attack_gap_base": 0.06,       # was a fixed 0.05 release
    "attack_gap_spread": 0.5,
    "attack_hesitate_prob": 0.06,  # chance of a longer "thinking" gap
    "attack_hesitate_range": (0.25, 0.7),

    # --- patrol variety, within bounds only (§5.3) --------------------------
    "turn_pause_prob": 0.5,        # brief stop at a turnaround
    "turn_pause_range": (0.15, 0.8),
    "mid_patrol_pause_prob": 0.01, # rare stop mid-leg (per shot)
    "mid_patrol_pause_range": (0.3, 1.5),

    # --- micro pauses between legs (§5.4) ------------------------------------
    "micro_pause_prob": 0.15,
    "micro_pause_range": (0.3, 2.0),

    # --- breaks + fatigue (§5.5) --------------------------------------------
    "break_enabled": True,
    "break_every_range": (8 * 60, 15 * 60),   # seconds between breaks
    "break_duration_range": (30, 120),        # seconds per break
    "fatigue_enabled": True,
    "fatigue_growth_per_hour": 0.15,          # reaction times & break freq grow

    # --- jittered skill/heal cadence (§5.6) ---------------------------------
    "skill_interval_range": (260, 340),       # 'a' skill, was a rigid 300s

    # --- rope-hang break (§6, opt-in / DEFENSIVE) ---------------------------
    "rope_enabled": False,         # keep OFF until the grab is tuned live (see §9)
    "rope_hang_prob": 0.4,         # chance a break parks on the rope vs stands idle
    # --- calibrated live 2026-08-09 (platform=(91,101), rope y 99..129 @ x~91) ---
    "ROPE_MINIMAP_X": 91,          # rope's minimap x (from calibrate_minimap_x)
    "ROPE_ALIGN_TOLERANCE": 2,     # minimap px; x reads 91-92, very stable
    "ROPE_ALIGN_MAX_ATTEMPTS": 12,
    "ROPE_FALL_TO_REGRAB_DELAY": 0.35,  # delay between down-jump and 2nd Down tap (TUNE)
    "ROPE_CLIMB_DURATION": 1.2,    # nominal climb time; exit is feedback-driven (TUNE)
    "ROPE_CLIMB_MAX": 4.0,         # hard cap on holding Up during exit (safety)
    "ROPE_EXIT_TOP_Y": 92,         # climb until y<=this (near rope top) before the
                                   # up-jump; live-proven value for this map's rope
    "ROPE_GRAB_MIN_DROP": 10,      # minimap px the char must DROP from its pre-grab y
                                   # to confirm a real grab (live: a grab drops ~25px)
    "ROPE_GRAB_X_TOL": 12,         # how far x may sit from the rope x after a grab
                                   # (char dot reads a few px different when hanging)
    "rope_hang_duration_range": (30, 120),
    "PLATFORM_Y": 101,             # real standing y on the farming platform (calibrated)

    # --- flourishes, OFF by default (§7) ------------------------------------
    "JUMP_KEY": Key.alt_l,         # LeftAlt = jump
    "idle_jump_prob": 0.0,
    "arrow_glance_prob": 0.0,      # do NOT enable: Up near a rope can grab it
}

# session-scoped state (fatigue clock)
_human_state = {"session_start": None}


def _human_session_reset(t):
    _human_state["session_start"] = t


def _fatigue_factor():
    """Grows slowly over a session so reaction times / break frequency drift up."""
    if not HUMAN["enabled"] or not HUMAN["fatigue_enabled"]:
        return 1.0
    s = _human_state.get("session_start")
    if not s:
        return 1.0
    hours = (time.time() - s) / 3600.0
    return 1.0 + HUMAN["fatigue_growth_per_hour"] * hours


def _rand_range(rng):
    lo, hi = rng
    return random.uniform(lo, hi)


# --- timing primitives ------------------------------------------------------
def human_sleep(base, spread=None, lo=0.0, hi=None, fatigue=False):
    """Gaussian-jittered, never-negative sleep. Respects F8 pause."""
    if not HUMAN["enabled"]:
        wait_with_pause(base)
        return base
    b = base * (_fatigue_factor() if fatigue else 1.0)
    if spread is None:
        spread = HUMAN["sleep_spread"]
    val = random.gauss(b, b * spread)
    val = max(lo, val)
    if hi is not None:
        val = min(hi, val)
    wait_with_pause(val)
    return val


def human_hold(key, base, spread=None, lo=0.03):
    """Jittered key-hold duration (replaces fixed click_press durations)."""
    if not HUMAN["enabled"]:
        click_press(key, base)
        return base
    if spread is None:
        spread = HUMAN["sleep_spread"]
    dur = max(lo, random.gauss(base, base * spread))
    click_press(key, dur)
    return dur


def _reaction_wait(rng):
    """Uniform pause in range, scaled by fatigue, keys assumed already released."""
    dur = _rand_range(rng) * _fatigue_factor()
    wait_with_pause(dur)
    return dur


# --- attack rhythm (§5.2) ---------------------------------------------------
def human_attack_during_walk(walk_key, attack_key, duration):
    """Like attack_during_walk but every shot's press/gap varies, with occasional
    hesitation and rare mid-leg pauses. Stays within the current patrol leg only."""
    global pause
    if not HUMAN["enabled"]:
        attack_during_walk(walk_key, attack_key, duration)
        return
    start = time.time()
    walking = False
    while time.time() - start < duration:
        if pause:
            safe_release_all()
            return
        if not walking:
            safe_press(walk_key)
            walking = True
        press = max(0.03, random.gauss(
            HUMAN["attack_press_base"],
            HUMAN["attack_press_base"] * HUMAN["attack_press_spread"]))
        safe_press(attack_key)
        time.sleep(press)
        safe_release(attack_key)
        if random.random() < HUMAN["attack_hesitate_prob"]:
            gap = _rand_range(HUMAN["attack_hesitate_range"])
        else:
            gap = max(0.02, random.gauss(
                HUMAN["attack_gap_base"],
                HUMAN["attack_gap_base"] * HUMAN["attack_gap_spread"]))
        time.sleep(gap)
        # rare mid-patrol pause (still on the same platform, just stop briefly)
        if random.random() < HUMAN["mid_patrol_pause_prob"]:
            safe_release(walk_key)
            _reaction_wait(HUMAN["mid_patrol_pause_range"])
            walking = False
    safe_release(walk_key)


# --- patrol variety helpers (§5.3, §5.4) ------------------------------------
def maybe_turn_pause():
    if not HUMAN["enabled"]:
        return
    if random.random() < HUMAN["turn_pause_prob"]:
        safe_release_all()
        _reaction_wait(HUMAN["turn_pause_range"])


def maybe_micro_pause():
    if not HUMAN["enabled"]:
        return
    if random.random() < HUMAN["micro_pause_prob"]:
        safe_release_all()
        _reaction_wait(HUMAN["micro_pause_range"])


def _idle_wait(duration):
    """Release everything and stand still for `duration`, staying pausable."""
    safe_release_all()
    end = time.time() + duration
    while time.time() < end:
        wait_with_pause(0.5)
    safe_release_all()


# --- breaks (§5.5) & rope-hang (§6) -----------------------------------------
def take_human_break(x_lo, x_hi, allow_rope=True):
    """Scheduled break: sometimes park on the rope, otherwise just stand idle."""
    dur = _rand_range(HUMAN["break_duration_range"]) * _fatigue_factor()
    safe_release_all()
    use_rope = (allow_rope and HUMAN["enabled"] and HUMAN["rope_enabled"]
                and HUMAN["ROPE_MINIMAP_X"] is not None
                and random.random() < HUMAN["rope_hang_prob"])
    if use_rope:
        rdur = _rand_range(HUMAN["rope_hang_duration_range"]) * _fatigue_factor()
        if rope_hang_break(rdur, x_lo, x_hi):
            return
        # rope bailed/failed -> fall through to a safe idle break
    print(f"[human] 休息 {int(dur)}s (idle)")
    _idle_wait(dur)


def rope_align(target_x, tol, max_attempts):
    """Tap Left/Right using minimap x feedback until within tol of target_x."""
    global pause
    for _ in range(max_attempts):
        if pause:
            return False
        x, y = get_character()
        if x < 0:
            human_sleep(0.2)
            continue
        if abs(x - target_x) <= tol:
            return True
        if x < target_x:
            click_press(Key.right, 0.08)
        else:
            click_press(Key.left, 0.08)
        human_sleep(0.15)
    x, y = get_character()
    return x >= 0 and abs(x - target_x) <= tol


def rope_grab():
    """Down + Jump to down-jump off the platform, short fall, tap Down to latch."""
    jump = HUMAN["JUMP_KEY"]
    safe_press(Key.down)
    human_sleep(0.18)
    safe_press(jump)
    human_sleep(0.10)
    safe_release(jump)
    human_sleep(HUMAN["ROPE_FALL_TO_REGRAB_DELAY"])  # let the character fall a bit
    safe_release(Key.down)
    human_sleep(0.05)
    safe_press(Key.down)                              # 2nd Down tap = latch on
    human_sleep(0.10)
    safe_release(Key.down)
    human_sleep(0.2)


def rope_exit():
    """Climb Up to the rope top, then up-jump onto the platform in ONE uninterrupted
    sequence (never pause on the rope — mobs knock you off). Live-proven 2026-08-09:
    climb to y<=ROPE_EXIT_TOP_Y (or until y stops decreasing = top), then tap Jump
    while Up is still held so it hops up onto the platform."""
    global pause
    jump = HUMAN["JUMP_KEY"]
    top_y = HUMAN["ROPE_EXIT_TOP_Y"]
    safe_press(Key.up)
    last_y, stable, t0 = None, 0, time.time()
    aborted = False
    while time.time() - t0 < HUMAN["ROPE_CLIMB_MAX"]:
        if pause:
            aborted = True
            break
        x, y = get_character()
        if y >= 0:
            if y <= top_y:                       # at platform level
                break
            if last_y is not None and y >= last_y:   # not climbing any higher = top
                stable += 1
                if stable >= 3:
                    break
            else:
                stable = 0
            last_y = y
        time.sleep(0.08)
    if not aborted:
        safe_press(jump)                          # up-jump (Up still held)
        human_sleep(0.15)
        safe_release(jump)
        human_sleep(0.12)
    safe_release(Key.up)
    human_sleep(0.2)


def rope_hang_break(dur, x_lo, x_hi):
    """Self-contained, reversible rope-hang. Returns True if it hung & recovered,
    False if it bailed safely (caller then does a normal idle break)."""
    target = HUMAN["ROPE_MINIMAP_X"]
    tol = HUMAN["ROPE_ALIGN_TOLERANCE"]

    # 1-2. align, or bail safely (never blind down-jump)
    if not rope_align(target, tol, HUMAN["ROPE_ALIGN_MAX_ATTEMPTS"]):
        print("[rope] 對不準繩子，改用原地休息")
        return False

    x0, y0 = get_character()
    # 3. grab, then verify via minimap: the character DROPPED a real distance from
    #    its pre-grab y (a down-jump onto the rope drops ~25px), and x is still near
    #    the rope. Relative drop is robust to the noisy absolute standing-y; the x
    #    check is loose because the char dot reads a few px different when hanging.
    rope_grab()
    x1, y1 = get_character()
    grabbed = ((x1 >= 0) and (y0 >= 0)
               and (y1 - y0 >= HUMAN["ROPE_GRAB_MIN_DROP"])
               and abs(x1 - HUMAN["ROPE_MINIMAP_X"]) <= HUMAN["ROPE_GRAB_X_TOL"])
    if not grabbed:
        # 4. grab failed -> recover to platform, panic-escape if still lost
        print(f"[rope] 抓繩失敗 (y {y0}->{y1}, x {x0}->{x1})，嘗試回平台")
        rope_exit()
        xf, yf = get_character()
        if xf < 0 or not (x_lo - 5 <= xf <= x_hi + 5):
            print("[rope] 位置異常，goto_freemarket")
            goto_freemarket()
        return False

    # 5. hang
    print(f"[rope] 掛繩休息 {int(dur)}s")
    _idle_wait(dur)

    # 6. return to platform, confirm bounds
    rope_exit()
    xf, yf = get_character()
    if xf < 0 or not (x_lo - 5 <= xf <= x_hi + 5):
        print("[rope] 回平台後位置異常，goto_freemarket")
        goto_freemarket()
    return True


def calibrate_minimap_x(samples=8, delay=0.4):
    """Stand on the rope once and run this to read its exact minimap x.
    Copy the printed value into HUMAN['ROPE_MINIMAP_X']."""
    print("站到繩子上，讀取小地圖 x ...")
    xs = []
    for i in range(samples):
        x, y = get_character()
        print(f"  sample {i + 1}: x={x}, y={y}")
        if x >= 0:
            xs.append(x)
        time.sleep(delay)
    if xs:
        xs_sorted = sorted(xs)
        med = xs_sorted[len(xs_sorted) // 2]
        print(f"建議 ROPE_MINIMAP_X = {med}  (min={min(xs)}, max={max(xs)})")
        return med
    print("沒讀到角色位置")
    return None


def _schedule_next_break(now):
    lo, hi = HUMAN["break_every_range"]
    interval = random.uniform(lo, hi) / _fatigue_factor()   # tire => break sooner
    return now + interval


# ============================================================================
# Human-like loops (shadow the originals above)
# ============================================================================
def blue_dragon_loop():
    exp_processor = ExpProcessor()
    global pause
    pause = True
    start_time = time.time()
    current_time = start_time
    skill_time = current_time - 300
    CHARACTER_Y = 91
    CHARACTER_X_L = 66
    CHARACTER_X_R = 161

    _human_session_reset(start_time)
    next_skill_gap = _rand_range(HUMAN["skill_interval_range"])
    next_break_at = _schedule_next_break(start_time)

    while True:
        if pause:
            time.sleep(0.1)
            continue

        # main stationary attack — jittered hold instead of a fixed 3s
        human_hold('c', base=3.0)

        end_time = time.time()
        current_state_time = (end_time - current_time)
        if current_state_time > 12 and not pause:
            walk_time = float(random.randint(20, 30)) / 10
            px, py = get_character()
            exp_gain = get_exp(exp_processor)
            if exp_gain != None:
                recent_exp_gain, exp_num_str, exp_pct_str = exp_gain[0], exp_gain[1], exp_gain[2]
                print(f"最近經驗:{recent_exp_gain}({exp_num_str}, {exp_pct_str}%)")
                if recent_exp_gain < 4000 and end_time - start_time > 180:
                    goto_freemarket()
                    continue
            if len(get_enemy()) > 0:
                print("有紅點!!!")
                goto_freemarket()
                continue

            # scheduled human break (idle or rope-hang) — released & recoverable
            if HUMAN["enabled"] and HUMAN["break_enabled"] and time.time() >= next_break_at:
                take_human_break(CHARACTER_X_L, CHARACTER_X_R, allow_rope=True)
                next_break_at = _schedule_next_break(time.time())
                current_time = time.time()
                continue

            # patrol (within bounds) with human rhythm + variety
            if abs(py - CHARACTER_Y) > 5 or (px - CHARACTER_X_L) <= 20:
                human_attack_during_walk(Key.right, 'c', walk_time)
                maybe_turn_pause()
            human_attack_during_walk(Key.left, 'c', walk_time + 0.25)
            maybe_turn_pause()
            human_hold(Key.right, base=0.1)
            human_sleep(0.5, fatigue=True)
            human_hold('h', base=0.1)          # heal / buff
            maybe_micro_pause()

            current_time = time.time()
            if current_time - skill_time > next_skill_gap:
                human_hold('a', base=0.5)      # skill on a jittered 260-340s cadence
                human_sleep(0.1)
                skill_time = current_time
                next_skill_gap = _rand_range(HUMAN["skill_interval_range"])


def egg_dragon_loop():
    exp_processor = ExpProcessor()
    global pause
    pause = True
    start_time = time.time()
    current_time = start_time
    skill_time = current_time - 300
    CHARACTER_Y = 91
    CHARACTER_X_L = 66
    CHARACTER_X_R = 161

    _human_session_reset(start_time)
    next_skill_gap = _rand_range(HUMAN["skill_interval_range"])
    next_break_at = _schedule_next_break(start_time)

    while True:
        if pause:
            time.sleep(0.1)
            continue

        human_hold('c', base=3.0)

        end_time = time.time()
        current_state_time = (end_time - current_time)
        if current_state_time > 15 and not pause:
            walk_time = float(random.randint(40, 50)) / 10
            exp_gain = get_exp(exp_processor)
            if exp_gain != None:
                recent_exp_gain, exp_num_str, exp_pct_str = exp_gain[0], exp_gain[1], exp_gain[2]
                print(f"最近經驗:{recent_exp_gain}({exp_num_str}, {exp_pct_str}%)")
                if recent_exp_gain < 4000 and end_time - start_time > 180:
                    goto_freemarket()
                    continue
            if len(get_enemy()) > 0:
                print("有紅點!!!")
                goto_freemarket()
                continue

            # scheduled human break — idle only (rope is a blue-map feature)
            if HUMAN["enabled"] and HUMAN["break_enabled"] and time.time() >= next_break_at:
                take_human_break(CHARACTER_X_L, CHARACTER_X_R, allow_rope=False)
                next_break_at = _schedule_next_break(time.time())
                current_time = time.time()
                continue

            human_attack_during_walk(Key.left, 'c', walk_time)
            maybe_turn_pause()
            human_attack_during_walk(Key.right, 'c', walk_time + 0.25)
            maybe_turn_pause()
            human_hold(Key.left, base=0.1)
            human_sleep(0.5, fatigue=True)
            human_hold('h', base=0.1)
            maybe_micro_pause()

            current_time = time.time()
            if current_time - skill_time > next_skill_gap:
                human_hold('a', base=0.5)
                human_sleep(0.1)
                skill_time = current_time
                next_skill_gap = _rand_range(HUMAN["skill_interval_range"])


In [ ]:
# ============================================================================
# INTEGRATED human-like run: farming + periodic breaks + auto fall-recovery.
# Run THIS cell (instead of the old blue_dragon_loop run cell) to start.
# Press F8 to begin / pause (starts paused).
#
# Design: recovery.farming_loop() owns movement/breaks/recovery (color-based
# minimap sensing, live-calibrated -- see recovery.py + HUMANIZE_HANDOFF.md).
# The notebook owns the screen-reading game logic (EXP OCR, red-dot/enemy) and
# passes it in as callbacks. EVERYTHING uses keyboard.py's controller + F8 pause
# (kb.pause) so there is ONE keyboard system (no stuck keys).
# ============================================================================
import time
import pyautogui
from pynput.keyboard import Listener
import keyboard as kb          # keyboard.py: safe_press/release_all, on_press, pause
import recovery                # recovery.py: farming_loop + calibrated recovery/drop
from exp_processor import ExpProcessor

_exp_proc = ExpProcessor()
_run_started = [None]          # set on first unpause, for the "180s then check EXP" rule


def _exp_check():
    """True -> EXP too low for too long (stuck) -> escape. Reuses notebook get_exp()."""
    if _run_started[0] is None:
        _run_started[0] = time.time()
    try:
        eg = get_exp(_exp_proc)                       # notebook fn (screen read, no keys)
    except Exception as e:
        print("exp read error:", e); return False
    if eg is not None and eg[0] is not None:
        return eg[0] < 4000 and (time.time() - _run_started[0]) > 180
    return False


def _enemy_check():
    """True -> another player on the minimap. Reuses notebook get_enemy()."""
    try:
        return len(get_enemy()) > 0                    # notebook fn (screen read, no keys)
    except Exception:
        return False


def _panic():
    """Free Market escape, kb-based (mirrors goto_freemarket) so it uses the SAME
    keyboard system as the loop. Then pauses; resume with F8."""
    kb.safe_release_all()
    print(f"[{time.time()}] 怕怕...自由市場GOGO")
    time.sleep(22)
    pyautogui.moveTo(1550, 2040, duration=0.5)
    pyautogui.click(x=1550, y=2040, clicks=2, interval=0.25, button='left')
    kb.pause = True


_listener = Listener(on_press=kb.on_press)   # F8 -> toggles kb.pause
_listener.start()
kb.pause = True                              # start paused; press F8 to begin
print("Integrated loop ready. Switch to the game and press F8 to start / pause.")
try:
    recovery.farming_loop(exp_check=_exp_check, enemy_check=_enemy_check, panic=_panic)
except KeyboardInterrupt:
    print("interrupted")
finally:
    kb.safe_release_all()
    _listener.stop()


In [40]:
listener = Listener(on_press=on_press)
listener.start() # 啟動監聽線程
try:
   blue_dragon_loop()
#    egg_dragon_loop()
except KeyboardInterrupt:
    print("程式被使用者中斷。")
finally:
    safe_release_all() # 確保程式結束時釋放所有按鍵
    listener.stop() # 停止監聽線程
    listener.join() # 等待監聽線程結束

[狀態切換] 繼續運行
最近經驗:0(11357256, 34.35%)
最近經驗:12200(11369456, 34.38%)
最近經驗:27450(11384706, 34.43%)
最近經驗:36600(11393856, 34.46%)
最近經驗:54900(11412156, 34.51%)
最近經驗:73200(11430456, 34.57%)
最近經驗:79300(11436556, 34.59%)
最近經驗:97600(11454856, 34.64%)
最近經驗:97600(11467056, 34.68%)
最近經驗:97600(11482306, 34.72%)
最近經驗:106750(11500606, 34.78%)
最近經驗:97600(11509756, 34.81%)
最近經驗:97600(11528056, 34.86%)
最近經驗:100650(11537206, 34.89%)
最近經驗:91500(11546356, 34.92%)
最近經驗:97600(11564656, 34.97%)
最近經驗:97600(11579906, 35.02%)
最近經驗:94550(11595156, 35.07%)
最近經驗:100650(11610406, 35.11%)
最近經驗:97600(11625656, 35.16%)
最近經驗:103700(11640906, 35.20%)
最近經驗:109800(11656156, 35.25%)
最近經驗:106750(11671406, 35.30%)
最近經驗:106750(11686656, 35.34%)
最近經驗:100650(11695806, 35.37%)
最近經驗:103700(11714106, 35.43%)
最近經驗:103700(11729356, 35.47%)
最近經驗:106750(11747656, 35.53%)
最近經驗:103700(11759856, 35.56%)
最近經驗:106750(11778156, 35.62%)
最近經驗:103700(11790356, 35.66%)
最近經驗:115900(11811706, 35.72%)
最近經驗:112850(11826956, 35.77%)
最近經驗:112850(1184220

In [31]:
window_title = 'MapleStory Worlds-Artale (?????)'

# 小地圖的固定座標 (需要手動測量) - 這些值你需要根據你的實際情況調整
MINIMAP_X_OFFSET = 20 # 根據你的螢幕截圖，小地圖左上角的x座標
MINIMAP_Y_OFFSET = 171 # 小地圖左上角的y座標
MINIMAP_WIDTH = 229   # 小地圖的寬度
MINIMAP_HEIGHT = 145  # 小地圖的高度

minimap_character_template_folder = 'assets/minimap_other_character/' 
character_centers_minimap = None
pil_img, np_img = capture_window_screenshot(window_title)
if np_img is None:
    print(f"找不到視窗 '{window_title}'，請確保遊戲正在運行並位於螢幕上。")
    time.sleep(2)
    # continue
minimap_end_x = MINIMAP_X_OFFSET + MINIMAP_WIDTH
minimap_end_y = MINIMAP_Y_OFFSET + MINIMAP_HEIGHT
minimap_end_x = min(minimap_end_x, np_img.shape[1])
minimap_end_y = min(minimap_end_y, np_img.shape[0])
minimap_img = np_img[MINIMAP_Y_OFFSET:minimap_end_y, MINIMAP_X_OFFSET:minimap_end_x]
character_centers_minimap = detect_character_on_minimap(minimap_img, 
                                                templates_folder=minimap_character_template_folder, 
                                                threshold=0.7, debug=True)

# character_centers_minimap = detect_red_dots(minimap_img, 
#                 templates_folder=minimap_character_template_folder, 
#                 threshold=0.75)
print(character_centers_minimap)

偵測到 0 個黃點，NMS前: 0
偵測到 0 個黃點，NMS前: 0
偵測到 0 個黃點，NMS前: 0
偵測到 4 個黃點，NMS前: 48
小地圖角色偵測結果圖已輸出：debug_output\minimap_character_detected.png，偵測到 4 個黃點，NMS前: 48
[[113, 74], [213, 73], [125, 33], [113, 28]]


In [13]:
exp_processor = ExpProcessor()
exp_gain, img = get_exp(exp_processor, True)
img